# RiskLens 01 · The CECL month-end run

This is the **traditional CECL run** a bank's allowance team does every month-end, before any AI is involved. RiskLens's agents never run these steps; they investigate the booked results through deterministic tools (step 5), so every number they quote traces back to this run.

| Step | What happens |
|---|---|
| 0 | Find the repo, load settings, make sure the seeded demo data exists |
| 1 | Source extracts: loans, borrowers, macro scenarios, scenario weights, overlays, model registry |
| 2 | Pre-run controls: data quality, model approval status, weights, macro coverage |
| 3 | Official run: PD / LGD / EAD by assigned model, lifetime ECL per scenario, probability weighting, overlays. Booked with a run log |
| 4 | Allowance movement analysis: waterfall, segments, macro variables, top borrowers, reconciliation |
| 5 | Hand-off: the tool schemas the agents get |

**Runs locally.** Start Jupyter from the repo root (`jupyter lab`) and run top to bottom. Data lives in `data/` (regenerated from a fixed seed, identical on every machine). No GPU and no model are needed here.

## 0 · Setup

In [ ]:
# RiskLens bootstrap: finds the repo checkout (local Jupyter) or clones it (Colab)
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Luizkauffmann/Nvidia-Redhat-RiskLens.git"

def _find_repo():
    env = os.environ.get("RISKLENS_REPO")
    for p in ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]:
        if (p / "config" / "gym.yaml").exists() and (p / "src" / "tools").exists():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    if not IN_COLAB:
        raise SystemExit("Start Jupyter from the Nvidia-Redhat-RiskLens checkout (or set RISKLENS_REPO).")
    REPO_DIR = Path("/content/Nvidia-Redhat-RiskLens")
    url = REPO_URL
    try:                                    # private repo: add a GITHUB_TOKEN Colab secret
        from google.colab import userdata
        url = f"https://{userdata.get('GITHUB_TOKEN').strip()}@github.com/Luizkauffmann/Nvidia-Redhat-RiskLens.git"
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt"), "matplotlib"], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.utils import config
config.reload()
from src.utils.notebook import M, ensure_demo_data, endpoint_up, load_dotenv, secret
load_dotenv()                                # local secrets live in .env (never committed)
print("repo   :", REPO_DIR, "| python", sys.version.split()[0], "| colab" if IN_COLAB else "| local")
print("data   :", ensure_demo_data())

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from src.data.loader import DataStore
from src.pipeline import cecl_run

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_colwidth", 80)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})
DATES = config.settings()["reporting_dates"]
PREV, CURR = DATES[-2], DATES[-1]
store = DataStore()
print("config :", config.config_dir())
print("data   :", config.path("generated"))
print("periods:", DATES)

## 1 · Source data
In a bank these extracts come from loan servicing, credit and finance systems. Here a seeded generator stands in for them (`python -m scripts.generate_demo_data` rebuilds them from scratch).

In [ ]:
display(pd.DataFrame(cecl_run.inventory(store)))
loans = store.loans.assign(date=store.loans.as_of_date.dt.strftime("%Y-%m-%d"))
book = loans.groupby(["portfolio", "date"]).agg(loans=("loan_id", "size"), balance=("current_balance", "sum")).unstack("date")
book.style.format({c: M for c in book.columns if c[0] == "balance"} | {c: "{:,}" for c in book.columns if c[0] == "loans"})

### Risk model registry
Each product is scored by a **PD**, an **LGD** and an **EAD** model. The registry holds versions and approval status; `config/models.yaml` holds the scorecard parameters the engine uses.

In [ ]:
display(store.model_metadata)
display(store.model_assignments_at(CURR).drop(columns=["as_of_date"]))
mp = store.model_parameters
mp[mp.model_id == "CRE_PD_v4"].set_index("parameter")[["value"]].T

### Macroeconomic scenarios (reasonable & supportable forecast, then reversion)

In [ ]:
macro = store.macro.assign(vintage=store.macro.as_of_date.dt.strftime("%Y-%m-%d"))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, var in zip(axes, ["unemployment_rate", "cre_price_growth"]):
    for (scen, vint), g in macro.groupby(["scenario", "vintage"]):
        ax.plot(g.forecast_quarter, g[var], marker="o", ls="-" if vint == CURR else "--", label=f"{scen} · {vint}")
    ax.set_title(f"{var}: July (dashed) vs August forecast")
    ax.tick_params(axis="x", rotation=45)
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.show()
display(store.weights.assign(as_of_date=store.weights.as_of_date.dt.strftime("%Y-%m-%d")))
display(store.overlays.assign(as_of_date=store.overlays.as_of_date.dt.strftime("%Y-%m-%d")))

## 2 · Pre-run controls
A FAIL blocks the official run, e.g. a loan scored with `CRE_PD_v5`, which is still in validation.

In [ ]:
val = {d: cecl_run.validate_inputs(d, store) for d in DATES}
print({d: v["status"] for d, v in val.items()})
pd.DataFrame({d: {c["check"]: c["status"] for c in v["checks"]} for d, v in val.items()})

## 3 · Official run
Per loan: lifetime ECL = Σ marginal PD × LGD × EAD over the remaining life, for each scenario path. Then probability-weight the scenarios and add the management overlays. CECL (lifetime) and IFRS 9 (12-month / lifetime by stage) are both booked.

In [ ]:
for d in DATES:
    t = time.time()
    out = cecl_run.run_official(d)
    for r in out["runs"]:
        print(f"{r['run_id']:26s} allowance {M(r['allowance']):>10}  ({r['loan_count']:,} loans, controls {r['validation_status']})  {time.time() - t:4.1f}s")
store = DataStore()            # re-read the booked results
store.run_log[["run_id", "executed_at_utc", "allowance", "scenario_set_id", "input_hash_loans", "validation_status"]].tail(4)

In [ ]:
r = store.results
r = r[r.framework == "CECL"].assign(date=lambda x: x.as_of_date.dt.strftime("%Y-%m-%d"))
by_port = r.groupby(["portfolio", "date"]).ecl.sum().unstack()
ax = (by_port[[PREV, CURR]] / 1e6).plot.bar(title="Modeled CECL ECL by portfolio ($M)", rot=0)
ax.set_ylabel("$M"); plt.show()
by_port["change"] = by_port[CURR] - by_port[PREV]
by_port.apply(lambda c: c.map(M))

### Databank export
A readable CSV copy for people (`data/outputs/databank/`): loans, borrowers and official results per date, plus reference tables and a data dictionary. The engine keeps reading Parquet.

In [ ]:
written = cecl_run.export_databank(config.path("outputs") / "databank", store, frameworks=["CECL", "IFRS9"])
pd.DataFrame(written).head(12)

## 4 · Allowance movement analysis
These cells use the **same deterministic tools the agents call** (`src/tools`), so the notebook, the agents and the NeMo Gym environment all see the same numbers. The first call computes the attribution walk and Shapley checks (1–2 minutes) and caches them in `data/outputs/cache`.

In [ ]:
from src.tools import ToolCallLog, call_tool, list_tools
LOG = ToolCallLog(config.path("outputs") / "notebook_tool_calls.jsonl")

def tool(name, **args):
    rec = call_tool(name, args, caller="workflow_notebook", log=LOG)
    if rec.status != "ok":
        raise RuntimeError(rec.error)
    return rec.result

cmp_ = tool("compare_allowance", previous_date=PREV, current_date=CURR)
print(f"Allowance {M(cmp_['previous_allowance'])} -> {M(cmp_['current_allowance'])}  change {M(cmp_['change'])} ({cmp_['change_pct']:.1f}%)")

In [ ]:
wf = tool("calculate_portfolio_attribution", previous_date=PREV, current_date=CURR)
order = list(dict.fromkeys(s["driver"] for s in wf["steps"]))
imp = {d["driver"]: d for d in wf["drivers"]}
labels = [PREV] + [imp[d]["label"] for d in order] + [CURR]
deltas = [imp[d]["impact"] for d in order]
heights = [wf["previous_allowance"]] + deltas + [wf["current_allowance"]]
bottoms = [0] + [wf["previous_allowance"] + sum(deltas[:i]) + min(deltas[i], 0) for i in range(len(deltas))] + [0]
colors = ["#7f8c8d"] + ["#c0392b" if v > 0 else "#27ae60" for v in deltas] + ["#2c3e50"]
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(range(len(labels)), [abs(h) / 1e6 for h in heights], bottom=[b / 1e6 for b in bottoms], color=colors)
for i, h in enumerate(heights):
    ax.text(i, (bottoms[i] + abs(h)) / 1e6 + 0.5, M(h), ha="center", fontsize=8)
ax.set_xticks(range(len(labels)), labels, rotation=20, ha="right")
ax.set_ylim(min(wf["previous_allowance"], wf["current_allowance"]) / 1e6 * 0.85, max(np.cumsum([wf["previous_allowance"]] + deltas)) / 1e6 * 1.05)
ax.set_title(f"CECL allowance waterfall (residual {M(wf['residual'])})"); ax.set_ylabel("$M")
plt.tight_layout(); plt.show()
pd.DataFrame(wf["drivers"])[["label", "impact", "share_of_change_pct"]]

In [ ]:
seg = tool("calculate_segment_attribution", previous_date=PREV, current_date=CURR)
mat = pd.DataFrame([{"portfolio": s["segment"], **s["drivers"], "total": s["total_change"]} for s in seg["segments"]]).set_index("portfolio")
grid = mat.drop(columns="total") / 1e6
lim = float(np.abs(grid.values).max())
fig, ax = plt.subplots(figsize=(10, 3.8))
im = ax.imshow(grid.values, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(grid.shape[1]), grid.columns, rotation=20, ha="right"); ax.set_yticks(range(grid.shape[0]), grid.index)
for (i, j), v in np.ndenumerate(grid.values):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)
ax.set_title("Allowance change by portfolio × driver ($M)"); fig.colorbar(im); plt.tight_layout(); plt.show()
mat.apply(lambda c: c.map(M))

In [ ]:
mv = tool("calculate_variable_attribution", previous_date=PREV, current_date=CURR)   # exact Shapley, 64 projections
v = pd.DataFrame(mv["variables"]).sort_values("impact")
ax = (v.set_index("variable")["impact"] / 1e6).plot.barh(title=f"Macro impact by variable (Shapley), total {M(mv['macro_total'])}")
ax.set_xlabel("$M"); plt.show()
v[["variable", "impact", "sequential_impact", "share_of_macro_pct"]]

In [ ]:
top = tool("identify_top_contributors", previous_date=PREV, current_date=CURR, n=10)
pd.DataFrame(top["contributors"])[["rank", "counterparty_id", "customer_name", "portfolio", "balance_current",
                                    "ecl_previous", "ecl_current", "ecl_change", "pd_12m_previous", "pd_12m_current",
                                    "main_driver", "reason_codes"]]

In [ ]:
mod = tool("compare_model_versions", previous_date=PREV, current_date=CURR)
w = tool("compare_scenario_weights", previous_date=PREV, current_date=CURR)
print("model versions changed:", mod["any_model_version_changed"], "| impact", M(mod["allowance_impact_of_model_change"]))
print("candidates not in production:", [m["model_id"] for m in mod["candidate_models_not_in_production"]])
print("scenario weights changed:", w["weights_changed"], "-> toward", w["weight_shifted_toward"], "| impact", M(w["allowance_impact_of_weight_change"]))

In [ ]:
rec = tool("reconcile_allowance_change", previous_date=PREV, current_date=CURR)
print(f"Reported {M(rec['reported_change'])} | explained {M(rec['explained_change'])} | residual {M(rec['residual'])} "
      f"| {rec['pct_explained']:.2f}% explained -> {rec['overall_status']}")
display(pd.DataFrame(rec["controls"])[["control", "status", "value"]])
pd.DataFrame(rec["order_robustness"]["drivers"])[["label", "sequential", "reverse_sequential", "shapley"]]

### What-if: scenario weights
Weighted ECL is linear in the weights, so this is an exact re-weighting of the booked scenario results. A what-if, not a booked run.

In [ ]:
wi = tool("run_scenario_analysis", as_of_date=CURR, weights={"baseline": 0.50, "adverse": 0.30, "severe_adverse": 0.20})
print("official:", M(wi["official_allowance"]), "| 50/30/20:", M(wi["custom"]["allowance"]), f"({wi['custom']['difference_pct']:+.1f}%)")
{k: M(v) for k, v in wi["single_scenario_allowance"].items()}

## 5 · Hand-off to the agents
Agents get **only these tool schemas**; each specialist sees a focused subset (`config/agents.yaml`). Every call is validated, repaired if needed (`"Aug 2026"` → `2026-08-31`) and logged with provenance. The NeMo Gym environment (notebook 03) exposes exactly these tools as HTTP endpoints.

In [ ]:
print({a: len(list_tools(a)) for a in ["orchestrator", "portfolio_agent", "credit_agent", "macro_agent", "model_risk_agent", "reconciliation_agent"]})
r = call_tool("calculate_pd_migration", {"previous_date": "July 2026", "current_date": "aug", "portfolio": "commercial real estate"}, caller="demo", log=LOG)
print("normalised parameters:", r.parameters)
print("provenance:", r.provenance())
bad = call_tool("calculate_pd_migration", {"previous_date": "2025-01-31", "current_date": CURR}, log=LOG)
print("structured error:", bad.error)
LOG.stats()

**Next:** `02_Agent_Investigator.ipynb` asks Nemotron to explain this movement; `03_NeMo_Gym_Environment.ipynb` scores it.

**Try a model change:** edit `config/models.yaml` (for example point CRE at `CRE_PD_v5` parameters), re-run steps 3–4, and watch the Model Risk tools and the reconciliation pick it up. `git checkout config/models.yaml` restores the defaults.